# Bayesian Directed Evolution

**BioPipelines example** — a data-driven directed-evolution loop on a protein–peptide interface. An SH3 domain is evolved to bind a proline-rich peptide more tightly: Boltz2 co-folds each variant complex, Prodigy predicts the interface ΔG, and a correlation → Bayesian-reweighting stack turns those energies into a smarter mutation distribution. Cycle 1 maps a small combinatorial library; cycle 2 tests the combinations the model proposes.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[colab]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines import Boltz2, Prodigy, Mutagenesis, MutationProfiler

with Pipeline(project="Setup", job="InstallTools"):
    Boltz2.install()
    Prodigy.install()           # protein–protein interface ΔG / Kd
    Mutagenesis.install()       # MutationEnv (shared by the profiler/composer/correlation stack)
    MutationProfiler.install()

## Cell 3 — Cycle 1: map a combinatorial library, learn the effects

We evolve a Grb2/Sem-5-type **SH3 domain** to bind a **proline-rich peptide** (the natural SH3 ligand class). The objective is the predicted interface free energy ΔG — *lower is tighter binding*.

1. **Mutagenesis** builds a **combinatorial** library over two binding-groove positions, drawing each from a small 4-residue alphabet (`mutate_to="ATKW"`, `combinatorial=True`). The combinatorial design is deliberate: it makes each amino acid recur at a position across several backgrounds, which is what gives the per-residue effect-size estimate its statistical footing (a one-variant-per-substitution scan would leave every group at n=1 and the correlation undefined).
2. **Boltz2** co-folds each variant with the peptide partner. `Bundle(Each(variants), peptide)` makes the peptide a constant chain in every complex (SH3 = chain A, peptide = chain B). The parent is folded once up front; **its MSA is recycled into every mutant** (`Mutagenesis(msas=parent)` → `Boltz2(msas=library)`), so the expensive MSA search runs once instead of per-variant — the short peptide chain folds single-sequence.
3. **Prodigy** predicts the interface ΔG between chains A and B from interfacial contacts.
4. **MutationProfiler** builds the prior amino-acid frequencies over the scanned positions.
5. **SequenceMetricCorrelation** measures, per position and amino acid, how each substitution shifts ΔG (a standardised effect size).
6. **BayesianAdjuster** performs a log-odds update (`mode="min"`: ΔG-lowering mutations boosted, ΔG-raising ones suppressed).
7. **MutationComposer** samples the next generation from the *adjusted* distribution.

Cells 3b/3c render the prior vs. adjusted profiles so you can see the reweighting move.

In [ ]:
# Cell 3: Cycle 1 — combinatorial library and Bayesian reweighting
from biopipelines.pipeline import *
from biopipelines import (Boltz2, Prodigy, Mutagenesis, MutationProfiler,
                          SequenceMetricCorrelation, BayesianAdjuster, MutationComposer)

with Pipeline(project="SH3Binder", job="BayesianEvolution"):
    Resources(gpu="A100", time="12:00:00", memory="16GB")

    # Grb2/Sem-5-type SH3 domain (the binder we evolve) and its proline-rich peptide ligand.
    sh3     = Sequence("GSPEEIIVVAKFDYVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMNKTGFVPSNYVERKN", ids="SH3")
    peptide = Sequence("VPPPVPPRRR", ids="pep")        # class-II PxxPxR motif

    # Fold the parent ONCE to generate its MSA, recycled into every mutant below.
    parent = Boltz2(proteins=sh3)

    # --- Cycle 1: combinatorial library at two groove positions, 4-residue alphabet. ---
    # combinatorial=True makes each (position, amino acid) recur across backgrounds,
    # so the per-residue effect sizes are estimated from n>1 (not a single variant each).
    Suffix("Cycle1")
    library = Mutagenesis(original=sh3, position="16+50", mode="specific",
                          mutate_to="ATKW", combinatorial=True, msas=parent)

    # Co-fold each variant with the constant peptide partner -> one A/B complex per variant.
    # Recycled per-mutant MSAs skip the server for the SH3 chain; the peptide folds single-sequence.
    scored  = Boltz2(proteins=Bundle(Each(library), peptide), msas=library)
    dg      = Prodigy(structures=scored, interface="A B")   # interface delta_g_kcal_mol per complex

    # Prior frequencies, per-mutation effect sizes on dG, then a Bayesian reweighting.
    profile     = MutationProfiler(original=sh3, mutants=library)
    correlation = SequenceMetricCorrelation(mutants=library.tables.sequences,
                                            data=dg.tables.affinity,
                                            original=sh3,
                                            metric="delta_g_kcal_mol")
    adjusted    = BayesianAdjuster(frequencies=profile.tables.absolute_frequencies,
                                   correlations=correlation.tables.correlation_2d,
                                   mode="min",       # lower delta_g_kcal_mol = tighter binding
                                   gamma=3.0,
                             positions="16+50")

    # Propose the next generation from the data-refined distribution.
    next_gen = MutationComposer(frequencies=adjusted.tables.absolute_probabilities,
                                num_sequences=12,
                                mode="weighted_random",
                                max_mutations=2)
next_gen.tables.sequences

### Cell 3b — Prior mutation profile

The amino-acid frequencies over the scanned positions *before* any ΔG information — a flat-ish prior reflecting the library design.

In [ ]:
# Prior profile (MutationProfiler sequence logos) — renders inline
profile

### Cell 3c — Bayesian-adjusted profile

The same positions *after* the log-odds update from the ΔG correlations. Residues that lowered ΔG (tighter binding) are boosted; those that raised it are suppressed — compare against the prior above.

In [ ]:
# Adjusted profile (BayesianAdjuster reweighted logos) — renders inline
adjusted

## Cell 4 — Cycle 2: test the proposed combinations, accumulate the signal

Cycle 1 mapped which residues at the two positions help. Now we close the loop: co-fold and score the combination variants the model proposed, then feed **both cycles** of data back into the correlation step so the distribution is refined on the larger, more diverse dataset.

1. **Boltz2 + Prodigy** score the cycle-2 combination library exactly as before.
2. **SequenceMetricCorrelation** is given the cycle-1 *and* cycle-2 mutants/ΔG together (list form) — more sequences per position means sharper, more reliable effect sizes.
3. **BayesianAdjuster** re-derives the distribution from the accumulated evidence.
4. A **Plot** compares the ΔG distribution of the starting single-mutant pool against the model-proposed combinations — the payoff of the loop, visible at a glance.

This is the engine of an iterative campaign: each cycle's data sharpens the next cycle's library.

In [ ]:
# Cell 4: Cycle 2 — score combinations and accumulate the correlation signal
from biopipelines import Plot

# The pipeline from Cell 3 is still active (on-the-fly), so we keep adding to it.
Suffix("Cycle2")

# Score the model-proposed combination variants.
scored2 = Boltz2(proteins=Bundle(Each(next_gen), peptide))
dg2     = Prodigy(structures=scored2, interface="A B")

# Re-estimate effects on the accumulated cycle-1 + cycle-2 data (list form).
correlation2 = SequenceMetricCorrelation(
    mutants=[library.tables.sequences, next_gen.tables.sequences],
    data=[dg.tables.affinity, dg2.tables.affinity],
    original=sh3,
    metric="delta_g_kcal_mol")
adjusted2 = BayesianAdjuster(frequencies=profile.tables.absolute_frequencies,
                             correlations=correlation2.tables.correlation_2d,
                             mode="min",
                             gamma=3.0,
                             positions="16+50")

# Did the proposed combinations bind tighter than the single-mutant pool?
Plot(Plot.Column(data=[dg.tables.affinity, dg2.tables.affinity],
                 y="delta_g_kcal_mol",
                 labels=["Cycle 1 (single mutants)", "Cycle 2 (proposed combos)"],
                 style="box",
                 title="Interface dG across evolution cycles",
                 ylabel="Prodigy dG [kcal/mol]  (lower = tighter)"))
dg2.tables.affinity